In [1]:
%AddJar file:/app/scala_lib/build/libs/robustInfer-scala-0.1.0.jar

Starting download from file:/app/scala_lib/build/libs/robustInfer-scala-0.1.0.jar
Finished download of robustInfer-scala-0.1.0.jar
Using cached version of robustInfer-scala-0.1.0.jar


In [2]:
import robustinfer._

In [3]:
import org.apache.spark.sql.{Dataset, Encoder, Encoders,SparkSession}
import org.apache.spark.sql.functions._
import scala.util.Random
import org.apache.spark.rdd.RDD

In [4]:
// manuall generate a small dataset
import breeze.linalg.{DenseVector, norm}

val smallData = Seq(
  Obs(i="1", x=Array(1.0, 2.0), y=1.0, timeIndex=Some(1), z=Some(1.0)),
  Obs(i="2", x=Array(2.0, 3.0), y=1.0, timeIndex=Some(1), z=Some(1.0)),
  Obs(i="3", x=Array(1.0, 2.0), y=0.0, timeIndex=Some(1), z=Some(0.0)),
  Obs(i="4", x=Array(2.0, 3.0), y=0.0, timeIndex=Some(1), z=Some(0.0)),
)

// repeat 10 times
// val mediumData = smallData.flatMap(obs => Seq.fill(10)(obs))

val df = smallData.toDS()

smallData = List(Obs(1,[D@633214b3,1.0,Some(1),Some(1.0)), Obs(2,[D@292f1e18,1.0,Some(1),Some(1.0)), Obs(3,[D@6f85f337,0.0,Some(1),Some(0.0)), Obs(4,[D@11295273,0.0,Some(1),Some(0.0)))
df = [i: string, x: array<double> ... 3 more fields]


[i: string, x: array<double> ... 3 more fields]

In [5]:
val drgu = new DRGU()
drgu.fit(df, maxIter = 20, tol = 1e-6, lambda = 0.1, dampingOnly = false, verbose = true)

Iteration 0: diff = 0.844447643301517
Iteration 1: diff = 0.2605654519868909
Iteration 2: diff = 0.0836008338166877
Iteration 3: diff = 0.03981630566128345
Iteration 4: diff = 0.017483679231298822
Iteration 5: diff = 0.00800491408977693
Iteration 6: diff = 0.0036010759921627044
Iteration 7: diff = 0.0016333155891462006
Iteration 8: diff = 7.381051880259202E-4
Iteration 9: diff = 3.341104081594727E-4
Iteration 10: diff = 1.5112464249460525E-4
Iteration 11: diff = 6.837989889372522E-5
Iteration 12: diff = 3.093532995552023E-5
Iteration 13: diff = 1.3996237780893065E-5
Iteration 14: diff = 6.332193566575239E-6
Iteration 15: diff = 2.864858959889251E-6
Iteration 16: diff = 1.2961328200873458E-6
Iteration 17: diff = 5.864041528460092E-7
Final step norm: 2.653041193303893E-7
Final parameter estimates:
Map(delta -> DenseVector(0.5811546150215456), beta -> DenseVector(5.684014305867641E-19, 8.724769467278655E-19, 1.4408001343466299E-18), gamma -> DenseVector(0.06567845386998009, 0.094547806775

drgu = robustinfer.DRGU@33f64dd0


true

In [6]:
drgu.summary().show()

+---------+--------------------+-------------------+--------------------+-------------------+
|parameter|            estimate|          std_error|             z_score|            p_value|
+---------+--------------------+-------------------+--------------------+-------------------+
|    delta|  1.0811546150215456| 0.2289591203253716|  2.5382461908294913|0.01114095811206961|
|   beta_0|5.684014305867641...|0.11161613812529658|5.092466377475768...|                1.0|
|   beta_1|8.724769467278655...|0.14636455848505944|5.960985061946714...|                1.0|
|   beta_2|1.440800134346629...|0.25798069661035633|5.584914504369900...|                1.0|
|  gamma_0| 0.06567845386998009|0.28952769966717473|  0.2268468749120739| 0.8205428127785286|
|  gamma_1| 0.09454780677501859|0.42966699309589884| 0.22004903400600787| 0.8258329668570616|
|  gamma_2| 0.16022626064499873| 0.7184166621988033|  0.2230269272355215| 0.8235145493634679|
|  gamma_3| 0.09454780677501859| 0.4296669930958987| 0.22004

In [7]:
val drguMiniBatch = new DRGU()
drguMiniBatch.fitMiniBatch(df, k=3, maxEpochs=20, pairsPerBatch = 10, lambda = 0.1, ema=0.0, m_variance = 3, verbose = true)

Epoch 0: diff norm = 0.844447643301517
Epoch 1: diff norm = 0.260565451986891
Epoch 2: diff norm = 0.0836008338166877
Epoch 3: diff norm = 0.03981630566128337
Epoch 4: diff norm = 0.01748367923129888
Epoch 5: diff norm = 0.008004914089776908
Epoch 6: diff norm = 0.0036010759921625947
Epoch 7: diff norm = 0.001633315589146198
Epoch 8: diff norm = 7.381051880259188E-4
Epoch 9: diff norm = 3.341104081595644E-4
Epoch 10: diff norm = 1.511246424946998E-4
Epoch 11: diff norm = 6.837989889360974E-5
Epoch 12: diff norm = 3.093532995530472E-5
Epoch 13: diff norm = 1.3996237780756015E-5
Epoch 14: diff norm = 6.332193566577794E-6
Epoch 15: diff norm = 2.8648589598628646E-6
Epoch 16: diff norm = 1.296132819886185E-6
Epoch 17: diff norm = 5.864041525853946E-7
Final step norm: 5.864041525853946E-7
Final parameter estimates:
Map(delta -> DenseVector(0.5811543729396426), beta -> DenseVector(2.9844802335283066E-19, 4.583576611307062E-19, 7.569442039771653E-19), gamma -> DenseVector(0.06567842958065735,

drguMiniBatch = robustinfer.DRGU@5c2bfad0


true

In [9]:
drguMiniBatch.summary().show(false)

+---------+----------------------+-------------------+---------------------+---------------------+
|parameter|estimate              |std_error          |z_score              |p_value              |
+---------+----------------------+-------------------+---------------------+---------------------+
|delta    |1.0811543729396424    |0.10421041820621485|5.57673966713804     |2.4506819684688708E-8|
|beta_0   |2.9844802335283066E-19|0.06444160739254681|4.631293902010094E-18|1.0                  |
|beta_1   |4.583576611307062E-19 |0.08450361724116981|5.424118825855378E-18|1.0                  |
|beta_2   |7.569442039771653E-19 |0.1489452246337162 |5.082030698457307E-18|1.0                  |
|gamma_0  |0.06567842958065735   |0.04919128757337448|1.3351638637775114   |0.18182272108337472  |
|gamma_1  |0.09454776743500881   |0.06961272646499772|1.3581965861163159   |0.17440131357301558  |
|gamma_2  |0.16022619701566618   |0.11722742008426397|1.366797946252629    |0.17168864671006867  |
|gamma_3  

In [19]:
val p = 1
val etaTrue = Array(0.0, 0.0)
val betaTrue = Array(0.0, 0.0, 0.0)

val numClusters = 500
val numObsPerCluster = 1

require(etaTrue.length == p + 1, "Length of eta must be p + 1")
require(betaTrue.length == p + 2, "Length of beta must be p + 2")

p = 1
etaTrue = Array(0.0, 0.0)
betaTrue = Array(0.0, 0.0, 0.0)
numClusters = 500
numObsPerCluster = 1


1

In [20]:
implicit val spark: SparkSession = SparkSession.builder()
  .appName("DRGU Console")
  .master("local[*]")
  .getOrCreate()

val df: Dataset[Obs] = UGEEUtils.generateData(numClusters, numObsPerCluster, p, etaTrue, betaTrue)

spark = org.apache.spark.sql.SparkSession@51989eb3
df = [i: string, x: array<double> ... 3 more fields]


[i: string, x: array<double> ... 3 more fields]

In [21]:
df.show(10)

+---+--------------------+--------------------+---------+---+
|  i|                   x|                   y|timeIndex|  z|
+---+--------------------+--------------------+---------+---+
| c1|[0.5424339474134525]| 0.10834973235545127|        1|0.0|
| c2|[0.48811393300999...| 0.26875503442054066|        1|1.0|
| c3|[0.03608753782292...| -0.6793656025659602|        1|1.0|
| c4|[0.7621799656781377]| -0.4982857595311496|        1|0.0|
| c5|[0.04169026119245...| 0.07383490800026599|        1|1.0|
| c6|[0.03171347243506...|-0.02624077731898...|        1|0.0|
| c7|[0.4023871775523268]|  -0.634831249123555|        1|1.0|
| c8| [0.924058786141048]|-0.00457623081889...|        1|1.0|
| c9|[0.6903271846407109]| 0.11921155210282719|        1|0.0|
|c10|[0.9842501557295041]| 0.43429650238428247|        1|1.0|
+---+--------------------+--------------------+---------+---+
only showing top 10 rows



In [22]:
val drgu = new DRGU()
drgu.fit(df, maxIter = 20, tol = 1e-6, lambda = 0.1, dampingOnly = false, verbose = true)

Iteration 0: diff = 0.01230787779726092
Iteration 1: diff = 5.564753915812759E-5
Iteration 2: diff = 1.3267040845353241E-6
Iteration 3: diff = 2.2067213186013053E-12
Final step norm: 1.2677968403852365E-16
Final parameter estimates:
Map(delta -> DenseVector(0.0011818091658156616), beta -> DenseVector(-0.011471501500187019, -0.004266051796361109), gamma -> DenseVector(3.164488877679042E-4, -8.243633128408241E-5, -8.327879462995458E-5))
Final variance estimate:
2.0037193003810083     -0.0035630924013746387  ... (6 total)
-0.003563092401374639  0.9859199694109515      ...
0.018754586860861622   0.4650730628374254      ...
5.019745088467227      -0.005980836508731458   ...
2.5554404351969744     -0.0033239551759864272  ...
2.544948074183602      -0.0033057447926543855  ...


drgu = robustinfer.DRGU@5301086a


true

In [23]:
drgu.summary().show()

+---------+--------------------+--------------------+--------------------+------------------+
|parameter|            estimate|           std_error|             z_score|           p_value|
+---------+--------------------+--------------------+--------------------+------------------+
|    delta|  0.5011818091658157| 0.06330433319103848| 0.01866869306164592|0.9851054032184985|
|   beta_0|-0.01147150150018...|0.044405404387550654|-0.25833570616920515|0.7961478317041979|
|   beta_1|-0.00426605179636...|0.026338602013041258| -0.1619695606565915|0.8713298243977836|
|  gamma_0|3.164488877679042E-4| 0.15872810952913968|0.001993653730940516|0.9984092955222459|
|  gamma_1|-8.24363312840824...| 0.08084013176988333|-0.00101974513746...|0.9991863612398801|
|  gamma_2|-8.32787946299545...|  0.0805091689869408|-0.00103440136915...| 0.999174667265057|
+---------+--------------------+--------------------+--------------------+------------------+



In [26]:
val drguMiniBatch = new DRGU()
drguMiniBatch.fitMiniBatch(df, k=499, maxEpochs=20, pairsPerBatch = 200000, lambda = 0.1, ema=0.0, s_variance = 500, m_variance = 499, verbose = true)

Epoch 0: diff norm = 0.012307877797260924
Epoch 1: diff norm = 5.564753915797971E-5
Epoch 2: diff norm = 1.3267040845499191E-6
Epoch 3: diff norm = 2.206814467522808E-12


drguMiniBatch = robustinfer.DRGU@6cca5a2


Final step norm: 2.206814467522808E-12
Final parameter estimates:
Map(delta -> DenseVector(0.0011818091658155842), beta -> DenseVector(-0.011471501500186993, -0.004266051796361284), gamma -> DenseVector(3.1644888776791034E-4, -8.243633128409743E-5, -8.327879462994764E-5))
Final variance estimate:
0.9556727399892835     0.006873884315080827    ... (6 total)
0.006873884315080827   0.5544216581692041      ...
0.0033491986520947843  0.2632549641634402      ...
0.12969874635189235    8.357795641020824E-4    ...
0.06689687102525183    -1.11594283411165E-4    ...
0.06694749546525088    -1.1199255697058265E-4  ...


true

In [27]:
drguMiniBatch.summary().show(false)

+---------+---------------------+---------------------+---------------------+------------------+
|parameter|estimate             |std_error            |z_score              |p_value           |
+---------+---------------------+---------------------+---------------------+------------------+
|delta    |0.5011818091658156   |0.043718937315293555 |0.027031973748414265 |0.9784342319804278|
|beta_0   |-0.011471501500186993|0.03329929903674262  |-0.34449678617947116 |0.7304727125547679|
|beta_1   |-0.004266051796361284|0.019007413973611262 |-0.22444146280414637 |0.8224138108501098|
|gamma_0  |3.1644888776791034E-4|0.006635989162281622 |0.047686769828766136 |0.9619658782548215|
|gamma_1  |-8.243633128409743E-5|0.0036394578118071155|-0.022650717647188488|0.9819288873572276|
|gamma_2  |-8.327879462994764E-5|0.0036418194105709793|-0.02286735975655939 |0.9817560767247682|
+---------+---------------------+---------------------+---------------------+------------------+

